# Reviewer 1, minor comment 3 — the rare-variant share of discovery over time

**Comment (verbatim).**

> Lines 99-101: It would be helpful to call out the increase in the contribution of rare variants
> to gene discovery. At the scale of the y axis, i.e. relative to common variants, it is difficult
> to see the increasing role of rare variants in discovery (assuming there is such). A
> supplementary figure, or addition to Fig. 1, that specifically shows that increase over time,
> would be a helpful contribution. As well as an addition to the discussion around what is leading
> to that increase (i.e. sample size, imputation, WGS, etc.).

Rare credible sets are 15,311 of 520,975 qualifying credible sets (2.94%), so on Figure 1c's shared
y-axis the rare band is a sliver and the trend is invisible. One figure showing the rare share on its
own percent scale is the whole answer. The drivers half of the comment (sample size, imputation, WGS)
is handled by Discussion wording, not by analysis.

**Scope is one figure. No analyses are added.**

## What is computed

For each metric and each year, from a single pre-computed table:

```
share = layer[layer_label == "rare"] / cumulative[tier_index == 4] * 100
```

i.e. the rare tier's increment over the all-common tier, as a fraction of the total. Both quantities
are already cumulative in the input, so there is no differencing and no other file is needed.

## Definition — label it, do not paraphrase it

The rare layer is entities **not reachable from any common-variant study** — those that would not
have been found without rare variants. It is a reachability difference between nested tiers, **not**
"genes first identified through rare variants". The published ancestry numbers (2,462 / 16,384) are
the same construction, and an earlier draft misdescribed them as first-discovery counts and had to be
corrected. The definition travels with every number: it is written into a `definition` column of the
output table.

## Input (no recomputation)

`data/intermediate_files/fig1c_cumulative_discovery_nested-r1.csv`, produced by
`chapters/06-review-r1/ancestry-mixed-split/01_ancestry_reclassification.ipynb`. Nesting order is
EUR → non-EUR → mixed (all common) → rare (any ancestry), `tier_index` 1–4.

**FinnGen R12 carries no publication date in the 25.06 release; this project pins it to 2024-11-04**,
following the ancestry notebook. That convention is kept unchanged, and it places FinnGen — the
largest single contributor of rare-variant credible sets — in the final year of the axis.

`MAX_YEAR = 2024`; 2025 is a partial year and is excluded throughout this project.

Rare credible sets pass extra QC by design — a rare credible set is retained only if it replicates,
or colocalises with a molQTL, or carries a protein-altering variant. The shares below are therefore
conservative. That is intended and already disclosed in the manuscript; it is neither relaxed nor
quantified here.

In [1]:
import pandas as pd

pd.set_option("display.width", 160)

DATA = "../../../data/intermediate_files/"
SUFFIX = "-r1"

MIN_YEAR, MAX_YEAR = 2006, 2024  # 2025 is a partial year and is excluded everywhere in this project
YEARS = list(range(MIN_YEAR, MAX_YEAR + 1))

METRICS = ["disease genes", "gene-disease pairs"]
PANEL = {"disease genes": "a", "gene-disease pairs": "b"}  # panel letters of Extended Data Fig. 10

nested = pd.read_csv(f"{DATA}fig1c_cumulative_discovery_nested{SUFFIX}.csv")

assert set(nested["metric"]) == set(METRICS), sorted(set(nested["metric"]))
assert nested["year"].min() == MIN_YEAR and nested["year"].max() == MAX_YEAR
print(f"nested rows: {len(nested):,}")
nested.head(3)

nested rows: 152


,metric,tier,tier_index,layer_label,year,count,cumulative,layer
0,disease genes,EUR (common),1,EUR common,2006,1,1,1.0
1,disease genes,EUR (common),1,EUR common,2007,40,41,41.0
2,disease genes,EUR (common),1,EUR common,2008,77,118,118.0


## 1. Rare share of cumulative discovery

`tier_index == 4` is the all-tiers row: `cumulative` is every entity discovered up to that year at
any allele frequency, and `layer` is the height of the rare band in Figure 1c — the entities
reachable only once rare variants are added. Their ratio is the plotted quantity.

In [2]:
def rare_share(metric: str) -> pd.DataFrame:
    """Rare-variant share of cumulative discovery, per year, for one metric.

    Args:
        metric: one of `disease genes`, `gene-disease pairs`.

    Returns:
        pandas.DataFrame: one row per year with the rare layer, the all-tiers cumulative total and
        the share in per cent.
    """
    tier4 = nested[(nested["metric"] == metric) & (nested["tier_index"] == 4)].sort_values("year")
    assert list(tier4["year"]) == YEARS
    assert (tier4["layer_label"] == "rare").all()
    assert (tier4["cumulative"] > 0).all()  # denominator is never zero, so the share is never undefined

    out = pd.DataFrame(
        {
            "metric": metric,
            "panel": PANEL[metric],
            "year": tier4["year"].to_numpy(),
            "rare_cumulative": tier4["layer"].to_numpy().astype(int),
            "all_variants_cumulative": tier4["cumulative"].to_numpy().astype(int),
        }
    )
    out["rare_share_pct"] = 100 * out["rare_cumulative"] / out["all_variants_cumulative"]
    return out


shares = pd.concat([rare_share(m) for m in METRICS], ignore_index=True)
for m in METRICS:
    print(f"### {m}")
    print(shares[shares["metric"] == m].drop(columns=["metric", "panel"]).to_string(index=False))
    print()

### disease genes
 year  rare_cumulative  all_variants_cumulative  rare_share_pct
 2006                0                        1        0.000000
 2007                0                       41        0.000000
 2008                0                      119        0.000000
 2009                0                      227        0.000000
 2010                0                      360        0.000000
 2011                1                      552        0.181159
 2012                2                      787        0.254130
 2013                3                      996        0.301205
 2014                4                     1152        0.347222
 2015                5                     1332        0.375375
 2016                6                     1525        0.393443
 2017               14                     1923        0.728029
 2018               21                     2629        0.798783
 2019               21                     3145        0.667727
 2020               26

### Early years, where the denominator is small

The denominator is the cumulative total, so it is never zero — the share is defined in every year of
the axis, and no year is dropped, imputed or clipped. It is nonetheless tiny at the start: 1 entity
in 2006, and the rare layer is exactly 0 until 2011, so 2006–2010 are true zeros rather than missing
data. Those five years are plotted as zero and the README says why. The `all_variants_cumulative`
column is exported so a reader can see the denominator behind every point.

In [3]:
def series_summary(metric: str) -> dict:
    """Descriptive summary of one plotted share series.

    Args:
        metric: one of `disease genes`, `gene-disease pairs`.

    Returns:
        dict: one row of the summary table.
    """
    s = shares[shares["metric"] == metric].reset_index(drop=True)
    diffs = s["rare_share_pct"].diff().dropna()
    decreased = s.loc[diffs[diffs < 0].index, "year"]
    first_nonzero = int(s.loc[s["rare_cumulative"] > 0, "year"].min())
    at_first = float(s.loc[s["year"] == first_nonzero, "rare_share_pct"].iloc[0])
    return {
        "metric": metric,
        "first_year_with_rare_layer": first_nonzero,
        "share_pct_at_first_year": at_first,
        "share_pct_2024": float(s["rare_share_pct"].iloc[-1]),
        "share_pct_change_first_to_2024": float(s["rare_share_pct"].iloc[-1] - at_first),
        "share_pct_ratio_2024_over_first": float(s["rare_share_pct"].iloc[-1] / at_first),
        "monotone_increasing": bool((diffs >= 0).all()),
        "n_years_share_decreased": int((diffs < 0).sum()),
        "years_share_decreased": ", ".join(map(str, decreased)) or "none",
        "rare_cumulative_2024": int(s["rare_cumulative"].iloc[-1]),
        "all_variants_cumulative_2024": int(s["all_variants_cumulative"].iloc[-1]),
        "min_denominator": int(s["all_variants_cumulative"].min()),
        "year_of_min_denominator": int(s.loc[s["all_variants_cumulative"].idxmin(), "year"]),
        "n_years_share_undefined": int(s["rare_share_pct"].isna().sum()),
    }


summary = pd.DataFrame([series_summary(m) for m in METRICS])
summary.T

,0,1
metric,disease genes,gene-disease pairs
first_year_with_rare_layer,2011,2011
share_pct_at_first_year,0.181159,0.138313
share_pct_2024,1.414688,1.7729
share_pct_change_first_to_2024,1.233529,1.634587
share_pct_ratio_2024_over_first,7.809079,12.818067
monotone_increasing,False,False
n_years_share_decreased,2,3
years_share_decreased,"2019, 2023","2016, 2018, 2019"
rare_cumulative_2024,115,630


## 2. Output table

One row per plotted point, carrying the numerator and the denominator so the share can be checked,
and a `definition` column so the file cannot be read without the wording the numbers require.

In [4]:
DEFINITION = (
    "rare_cumulative: entities not reachable from any common-variant study - those that would not "
    "have been found without rare variants. A reachability difference between nested tiers (the rare "
    "band of Figure 1c), NOT a count of entities first identified through rare variants. "
    "all_variants_cumulative: all entities discovered up to that year at any allele frequency. "
    "rare_share_pct = 100 x rare_cumulative / all_variants_cumulative, the plotted quantity."
)

tidy = shares.copy()
tidy["definition"] = DEFINITION
tidy = tidy.sort_values(["metric", "year"], kind="stable").reset_index(drop=True)

assert len(tidy) == len(METRICS) * len(YEARS)
assert tidy["rare_share_pct"].notna().all()

out_csv = f"{DATA}rare_discovery_over_time{SUFFIX}.csv"
tidy.to_csv(out_csv, index=False)
print(f"wrote {out_csv}  ({len(tidy)} rows)")

summary_csv = f"{DATA}rare_discovery_over_time_summary{SUFFIX}.csv"
summary.to_csv(summary_csv, index=False)
print(f"wrote {summary_csv}")
tidy.head(3)

wrote ../../../data/intermediate_files/rare_discovery_over_time-r1.csv  (38 rows)
wrote ../../../data/intermediate_files/rare_discovery_over_time_summary-r1.csv


,metric,panel,year,rare_cumulative,all_variants_cumulative,rare_share_pct,definition
0,disease genes,a,2006,0,1,0.0,rare_cumulative: entities not reachable from a...
1,disease genes,a,2007,0,41,0.0,rare_cumulative: entities not reachable from a...
2,disease genes,a,2008,0,119,0.0,rare_cumulative: entities not reachable from a...


## 3. Sentences for the response letter

Every number below comes from the two exported tables, not from a bare `print()`.

In [5]:
for _, r in summary.iterrows():
    print(f"{r['metric']}:")
    print(
        f"  2024: {r['rare_cumulative_2024']:,} of {r['all_variants_cumulative_2024']:,} "
        f"({r['share_pct_2024']:.2f}%) would not have been found without rare variants"
    )
    print(
        f"  share rises from {r['share_pct_at_first_year']:.2f}% in "
        f"{r['first_year_with_rare_layer']} to {r['share_pct_2024']:.2f}% in 2024 "
        f"({r['share_pct_ratio_2024_over_first']:.1f}x)"
    )
    print(f"  monotone increasing: {r['monotone_increasing']} (decreases in {r['years_share_decreased']})")
    print(
        f"  smallest denominator {r['min_denominator']:,} entities in "
        f"{r['year_of_min_denominator']}; years with an undefined share: "
        f"{r['n_years_share_undefined']}"
    )
    print()

disease genes:
  2024: 115 of 8,129 (1.41%) would not have been found without rare variants
  share rises from 0.18% in 2011 to 1.41% in 2024 (7.8x)
  monotone increasing: False (decreases in 2019, 2023)
  smallest denominator 1 entities in 2006; years with an undefined share: 0

gene-disease pairs:
  2024: 630 of 35,535 (1.77%) would not have been found without rare variants
  share rises from 0.14% in 2011 to 1.77% in 2024 (12.8x)
  monotone increasing: False (decreases in 2016, 2018, 2019)
  smallest denominator 1 entities in 2006; years with an undefined share: 0



## 4. Rendering the figure

`ed10_rare_variant_discovery.R` in this directory draws the figure and reads only
`rare_discovery_over_time-r1.csv`. Keeping every number in the exported table means the figure, this
notebook and the response letter cannot disagree. See `README.md` for how to run it.